In [8]:
import pandas as pd

articles = pd.read_csv('./articles.csv')
customers = pd.read_csv('./customers.csv')
transactions = pd.read_csv('./transactions_train.csv')

In [9]:
articles['text'] = (
    articles['prod_name'] + ' ' +
    articles['product_type_name'] + ' ' +
    articles['colour_group_name'] + ' ' +
    articles['garment_group_name'] + ' ' +
    articles['detail_desc'].fillna('')
)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer('all-MiniLM-L6-v2',device='cuda')
all_texts = articles['text'].tolist()
#all_embeddings = st_model.encode(all_texts, show_progress_bar=True, batch_size=256)
#np.save('article_embeddings.npy', all_embeddings)
all_embeddings = np.load('article_embeddings.npy')

Batches:   0%|          | 0/413 [00:00<?, ?it/s]

In [11]:
import faiss
dimension = all_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(all_embeddings.astype('float32'))


######여기까지 임베딩 벡터########

In [6]:
query = "casual summer white t-shirt"
query_embedding = st_model.encode([query])

D, I = index.search(query_embedding.astype('float32'), k=10)

# 중복 제거
seen = set()
results = []
for idx in I[0]:
    prod_name = articles.iloc[idx]['prod_name']
    if prod_name not in seen:
        seen.add(prod_name)
        results.append(idx)
    if len(results) == 5:
        break

print("검색 결과 (중복 제거):")
for i, idx in enumerate(results):
    print(f"{i+1}. {articles.iloc[idx]['prod_name']} - {articles.iloc[idx]['detail_desc']}")

검색 결과 (중복 제거):
1. Summer price tee - T-shirt in soft cotton jersey with a print motif.
2. SG Summer Top - Wide-fitting sports top in fast-drying functional fabric with a print motif on the front, cap sleeves and a longer, sewn-in vest top with a racer back.
3. SUMMER Top - Short, wide, sports top in printed, fast-drying mesh layered over a vest top with a racer back.
4. Summer top - Short, wide, sports top in printed, fast-drying mesh layered over a vest top with a racer back.
5. Summer graphic tee - T-shirt in soft jersey with a motif on the front.


In [ ]:
# 구매 많은 유저 한 명 뽑기
top_user = transactions['customer_id'].value_counts().index[0]
user_history = transactions[transactions['customer_id'] == top_user]['article_id'].tolist()

유저 구매 아이템 수: 1895
구매한 아이템 예시:
- Skirt Pencil Stretch.
- Jafar
- Skirt Pencil Midi


In [ ]:
# 유저 히스토리 임베딩 평균
article_id_to_idx = {aid: idx for idx, aid in enumerate(articles['article_id'])}

user_embeddings = []
for aid in user_history:
    if aid in article_id_to_idx:
        idx = article_id_to_idx[aid]
        user_embeddings.append(all_embeddings[idx])

user_vector = np.mean(user_embeddings, axis=0, keepdims=True)

# 추천
D, I = index.search(user_vector.astype('float32'), k=10)

seen = set(user_history)
results = []
for idx in I[0]:
    aid = articles.iloc[idx]['article_id']
    if aid not in seen:
        results.append(idx)
    if len(results) == 5:
        break

print("유저 맞춤 추천:")
for i, idx in enumerate(results):
    print(f"{i+1}. {articles.iloc[idx]['prod_name']} - {articles.iloc[idx]['detail_desc']}")
    
    ##############################

유저 맞춤 추천:
1. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.
2. Twiggy - Calf-length dress in mesh with a low-cut back and long sleeves. Seam at the waist, a gently flared skirt and raw edges at the cuffs and hem. Jersey lining.
3. LASSIE LINEN L/S - Long-sleeved top in a linen weave with a V-neck. Slightly longer at the back.
4. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.
5. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.


In [ ]:
seen_names = set()
seen_ids = set(user_history)
results = []
for idx in I[0]:
    aid = articles.iloc[idx]['article_id']
    name = articles.iloc[idx]['prod_name']
    if aid not in seen_ids and name not in seen_names:
        seen_names.add(name)
        results.append(idx)
    if len(results) == 5:
        break

print("유저 맞춤 추천 (중복 제거):")
for i, idx in enumerate(results):
    print(f"{i+1}. {articles.iloc[idx]['prod_name']} - {articles.iloc[idx]['detail_desc']}")

유저 맞춤 추천 (중복 제거):
1. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.
2. Twiggy - Calf-length dress in mesh with a low-cut back and long sleeves. Seam at the waist, a gently flared skirt and raw edges at the cuffs and hem. Jersey lining.
3. RAVEN halfzip ls - Fitted running top in fast-drying functional fabric with a stand-up collar and zip at the top. Long sleeves with thumbholes at the cuffs, a small zipped pocket at the back, reflective details and a gently rounded hem. Slightly longer at the back. Soft brushed inside.
4. HEATHER halfzip ls - Fitted running top in fast-drying functional fabric with an elasticated hood and a zip at the top. Long raglan sleeves, a small zipped pocket at the back, reflective details and a gently rounded hem. Slightly longer at the back. Soft brushed inside.
5. ELENA baselayer - Base layer tights in a soft wool blend with an elasticated waist.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "./qwen"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

모델 로드 완료!


In [ ]:
def rerank_with_llm(query, candidates, top_k=3):
    candidate_text = "\n".join([
        f"{i+1}. {articles.iloc[idx]['prod_name']}: {articles.iloc[idx]['detail_desc']}"
        for i, idx in enumerate(candidates)
    ])
    
    prompt = f"""당신은 패션 추천 전문가입니다. 반드시 한국어로만 답변하세요. 한자나 중국어는 절대 사용하지 마세요.

사용자 검색어: "{query}"

후보 상품:
{candidate_text}

위 후보 중 가장 적합한 {top_k}개를 선택하고 이유를 설명하세요.
형식:
1. [상품 번호] - [이유]
2. [상품 번호] - [이유]
3. [상품 번호] - [이유]"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(qwen_model.device)
    outputs = qwen_model.generate(**inputs, max_new_tokens=200)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

1. 1 - [W GARDA SKIRT EQ] - 이 상품은 키높이의 디테일과 detachable tie belt로 인해 다양한 스타일링이 가능하며, 일반적인 길이와 디테일로 일상에서 쉽게 매치할 수 있습니다.
2. 2 - [Mia l/s top] - 이 상품은 편안하면서도 시크한 느낌을 주는 소재와 디테일로, 일상적인 의상과 잘 어울릴 것입니다.
3. 4 - [Singapore: Calf-length skirt] - 이 상품은 치마처럼 긴 길이와 부드러운 질감으로, 따뜻한 날씨에 걸칠 수 있으며, 다른 높이의 상의와 함께 매치하기 좋습니다.


In [ ]:
query = "나한테 잘 맞는 옷"
query_embedding = st_model.encode([query])
D, I = index.search(query_embedding.astype('float32'), k=10)
result = rerank_with_llm(query, I[0][:5])
print(result)

In [95]:
user_vector = np.mean(user_embeddings, axis=0, keepdims=True)

D, I = index.search(user_vector.astype('float32'), k=20)

seen_names = set()
seen_ids = set(user_history)
candidates = []
for idx in I[0]:
    aid = articles.iloc[idx]['article_id']
    name = articles.iloc[idx]['prod_name']
    if aid not in seen_ids and name not in seen_names:
        seen_names.add(name)
        candidates.append(idx)
    if len(candidates) == 5:
        break

result = rerank_with_llm("이 유저의 구매 히스토리 기반 추천", candidates)
print(result)

1. 1 - [LASSIE LINEN L/S] - 이 유저의 구매 히스토리에서 라인 넷과 반원형 바지가 많이 구매되었고, 이 상품은 유아복 스타일의 롱슬리브 티셔츠로, 편안함과 착용감을 제공하여 아이들의 활동성을 돕기 좋습니다.
2. 3 - [RAVEN halfzip ls] - [HEATHER halfzip ls] - 이 두 상품 모두 빠르게 말리는 기능성 소재와 반바지 형태, 긴 손목 스트랩 및 반짝임 디테일 등이 유저의 구매 패턴에 잘 맞아 떨어집니다. 특히 빠른 말리는 특성으로 운동이나 활동 시 활용도가 높을 것으로 예상됩니다.
3. 5 - [ELENA baselayer] - 이 유저의 구매 히스토리에서 터틀넥과 반원형 바지가 주요 구매 트렌드였으므로, 이 상품은 기능성과 유아복 스타일을 동시에 만족시켜주며, 일상생활에서 활용도가 높을 것으로 보입니다.


In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch
from datasets import load_dataset
dataset = load_dataset("beomi/KoAlpaca-v1.1a")
def format_data(item):
    return {"text": f"### 지시사항:\n{item['instruction']}\n\n### 답변:\n{item['output']}"}

train_dataset = dataset['train'].map(format_data, remove_columns=['instruction', 'output', 'url'])

tokenizer = AutoTokenizer.from_pretrained("./qwen")
tokenizer.pad_token = tokenizer.eos_token

# 토크나이징
def tokenize(item):
    result = tokenizer(
        item["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = train_dataset.map(tokenize, remove_columns=["text"])
print(tokenized_dataset)

c:\Users\swson23\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 21155
})


In [ ]:
import torch
from peft import LoraConfig, get_peft_model

qwen_model = AutoModelForCausalLM.from_pretrained(
    "./qwen",
    torch_dtype=torch.float16,
    device_map="cuda"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

#qwen_model = get_peft_model(qwen_model, lora_config)
#qwen_model.config.use_cache = False
#qwen_model.print_trainable_parameters()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 1,843,200 || all params: 3,087,781,888 || trainable%: 0.0597


In [8]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./qwen_korean",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=100,
    save_steps=500,
    warmup_steps=100,
    report_to="none",
    save_safetensors=False
)

trainer = Trainer(
    model=qwen_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

#trainer.train()

In [ ]:
#qwen_model.save_pretrained("./qwen_korean")
#tokenizer.save_pretrained("./qwen_korean")

저장 완료!


In [13]:
# 1. 모델 로드
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("./qwen")
base_model = AutoModelForCausalLM.from_pretrained(
    "./qwen",
    torch_dtype=torch.float16,
    device_map="cuda"
)
korean_model = PeftModel.from_pretrained(base_model, "./qwen_korean")
#korean_model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [14]:
#del qwen_model
del base_model
torch.cuda.empty_cache()

In [15]:
def rerank_with_llm(query, candidates, top_k=3):
    candidate_text = "\n".join([
        f"{i+1}. 상품명: {articles.iloc[idx]['prod_name']}, 카테고리: {articles.iloc[idx]['product_type_name']}, 색상: {articles.iloc[idx]['colour_group_name']}, 설명: {articles.iloc[idx]['detail_desc'][:80]}"
        for i, idx in enumerate(candidates)
    ])
    
    prompt = f"""당신은 패션 추천 전문가입니다. 반드시 한국어로만 답변하세요. 한자나 중국어는 절대 사용하지 마세요.

사용자 검색어: "{query}"

후보 상품:
{candidate_text}

위 후보 중 가장 적합한 {top_k}개를 선택하고 이유를 설명하세요.
형식:
1. [상품 번호] - [이유]
2. [상품 번호] - [이유]
3. [상품 번호] - [이유]"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(korean_model.device)
    outputs = korean_model.generate(**inputs, max_new_tokens=500)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response


In [16]:
query = "나한테 잘 맞는 옷"
query_embedding = st_model.encode([query])
D, I = index.search(query_embedding.astype('float32'), k=10)
result = rerank_with_llm(query, I[0][:5])
print(result)

1. [1] - 빨간색 스커트는 허리 높이에 탈착 가능한 티가 있어서, 허리가 좀 더 낮은 분에게도 어울릴 수 있습니다. 또한 셔츠와 함께 착용하면 데일리룩으로도 좋습니다.
2. [4] - 검정색 스카트는 겨울철에 어울리는 브랜드의 스카트 중 하나입니다. 허리 높이가 높아서 허리가 좀 더 높은 분에게도 어울릴 수 있으며, 턱시도 같은 느낌의 스커트라서 겨울철에 더욱 어울릴 것입니다.
3. [3] - 회색 브라운색 레이스 니트셔츠는 데일리룩에서 활용하기 좋을 것 같습니다. 흰색과 연관된 색상으로, 데일리룩에서도 어울릴 수 있는 색상입니다. 이 레이스 니트셔츠는 키즈 라인에서도 사용되고 있어서, 아이들과 함께 나들이를 할 때도 좋을 것입니다.
